* It should be noted that price and volume have a persumably significant, but very low correlation (~0.1). This will not be further elaborated

# Init

In [ ]:
%run ./preprocessing/init.ipynb

# Plot

In [ ]:
assets = [
    "bitcoin_trading_metadata",
    "nasdaq_trading_metadata",
    "snp_trading_metadata",
    "dow_jones_trading_metadata",
    "gold_trading_metadata",
    "oil_trading_metadata"
]

results = dict()

for asset in assets:
    with psycopg.connect(CONNECTION_STRING) as conn:
        res = conn.execute(f"""
            select 
                dt as "date",
                trading_volume / outstanding_supply as turnover_rate
            from {asset}
        """)
        frame = pd.DataFrame(res, columns=["date", "turnover_rate"])
        frame["date"] = pd.to_datetime(frame["date"])
        srs = frame.set_index("date").sort_index().squeeze()
        results[asset] = srs

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.transforms import blended_transform_factory
import pandas as pd
from math import log

STYLE = {
    "bitcoin_trading_metadata": ("Bitcoin", "#ff0000", 1.0),
    "nasdaq_trading_metadata": ("NASDAQ", "#0092BC", 0.7),
    "snp_trading_metadata": ("S&P 500", "#2F4F4F", 0.7),
    "dow_jones_trading_metadata": ("Dow Jones", "#62250b", 0.7),
    "gold_trading_metadata": ("Gold", "#e88f09", 0.7),
    "oil_trading_metadata": ("Oil", "#000000", 0.7),
}

START = pd.Timestamp("2010-01-01")
END   = pd.Timestamp("2025-07-01")

LINEWIDTH = 2.5
FONT_SCALE = 1.3

TITLE_SIZE = 16 * FONT_SCALE
LABEL_SIZE = 10 * FONT_SCALE
FOOTNOTE_SIZE = TITLE_SIZE * 0.7

FIG_NUMBERS = [8,9,10]

CHART_SPECS = [
    {
        "assets": list(STYLE.keys()),
        "title": "Trading intensity of assets compared",
        "ylabel": "Trading intensity",
        "log_transform": False,
        "percent": False,
        "fig_n": FIG_NUMBERS[0],
        "drop_labels": {
            "nasdaq_trading_metadata",
            "snp_trading_metadata",
            "dow_jones_trading_metadata"
        }
    },
    {
        "assets": [
            "bitcoin_trading_metadata",
            "nasdaq_trading_metadata",
            "snp_trading_metadata",
            "dow_jones_trading_metadata",
        ],
        "title": "Trading intensity of assets except oil and gold",
        "ylabel": "Trading intensity",
        "log_transform": False,
        "percent": False,
        "fig_n": FIG_NUMBERS[1],
        "drop_labels": {
            "nasdaq_trading_metadata",
            "snp_trading_metadata",
            "dow_jones_trading_metadata"
        }
    },
    {
        "assets": list(STYLE.keys()),
        "title": "Trading intensity of assets compared",
        "ylabel": "ln( Trading intensity )",
        "log_transform": True,
        "percent": False,
        "fig_n": FIG_NUMBERS[2],
        "drop_labels": set()
    }
]

plt.rcParams["font.family"] = "Times New Roman"

In [ ]:
def transform_series(series, use_log, percent):
    s = series.loc[:END].dropna()
    if percent:
        s = s * 100
    if use_log:
        s = s.map(log)
    return s


def add_right_labels(ax, spec):
    trans = blended_transform_factory(ax.transAxes, ax.transData)

    labels = []

    for asset in spec["assets"]:
        if asset in spec["drop_labels"]:
            continue

        name, color, _ = STYLE[asset]
        s = transform_series(results[asset], spec["log_transform"], spec["percent"])

        labels.append((name, color, float(s.iloc[-1])))

    # --- add "Other assets" anchored to S&P when labels were dropped ---
    if "snp_trading_metadata" in spec["drop_labels"]:
        name, color, _ = STYLE["snp_trading_metadata"]
        s = transform_series(results["snp_trading_metadata"], spec["log_transform"], spec["percent"])
        labels.append(("Other assets", color, float(s.iloc[-1])))
    # ---------------------------------------------------------------

    labels.sort(key=lambda x: x[2])

    gap = (labels[-1][2] - labels[0][2]) * 0.03 if len(labels) > 1 else 0.05
    adjusted = []

    for name, color, y in labels:
        y = max(y, adjusted[-1][2] + gap) if adjusted else y
        adjusted.append((name, color, y))

    for name, color, y in adjusted:
        ax.text(
            1.01, y, name,
            transform=trans,
            ha="left",
            va="center",
            color=color,
            fontsize=LABEL_SIZE
        )

In [ ]:
fig, axes = plt.subplots(
    3, 1,
    figsize=(14,21),
    facecolor="#ffffff"
)

major = pd.date_range("2010-01-01","2025-01-01",freq="YS")
minor = pd.date_range("2010-01-01","2025-07-01",freq="QS-JAN")

for ax, spec in zip(axes, CHART_SPECS):

    ax.set_facecolor("#ffffff")
    ax.grid(False)

    for asset in spec["assets"]:
        name, color, alpha = STYLE[asset]
        s = transform_series(results[asset], spec["log_transform"], spec["percent"])

        ax.plot(
            s.index,
            s.values,
            color=color,
            alpha=alpha,
            linewidth=LINEWIDTH
        )

    ax.set_xlim(START, END)
    ax.set_xticks(major)
    ax.set_xticks(minor, minor=True)

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.tick_params(axis="x", which="major", length=6, labelsize=LABEL_SIZE)
    ax.tick_params(axis="x", which="minor", length=3)

    ax.set_ylabel(spec["ylabel"], fontsize=LABEL_SIZE)
    ax.set_title(spec["title"], fontsize=TITLE_SIZE, color="black")

    add_right_labels(ax, spec)

    ax.text(
        0.5,
        -0.1,
        f"Figure {spec['fig_n']}",
        transform=ax.transAxes,
        style="italic",
        ha="center",
        va="top",
        fontsize=FOOTNOTE_SIZE,
        color="black"
    )

plt.tight_layout(rect=[0,0.02,0.98,0.98], h_pad=3.0)

fig.savefig("../../figures/turnover_rate_compared.jpg", format="jpg", dpi=800, bbox_inches="tight")
pass